In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [3]:
df = pd.read_csv("data/kickstarter_clean.csv")

In [4]:
X = df.drop(columns='State', axis=1)
y = df['State']

In [5]:
RSEED = 42

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=RSEED)

In [ ]:
param_grid = {
    "learning_rate": [0.04, 0.05, 0.06],
    "max_depth": [5, 6],
    "subsample": [0.75, 0.8, 0.85],
    "colsample_bytree": [0.75, 0.8, 0.85],
    "min_child_weight": [1, 2],
    "gamma": [0, 0.05],
    "reg_alpha": [0.05, 0.1, 0.2],
    "reg_lambda": [1.2, 1.5, 1.8],
    "scale_pos_weight": [1, 1.2, 1.5]
}

In [ ]:
xgb = XGBClassifier(
    n_estimators=1200,
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False,
    n_jobs=-1
)

In [ ]:
grid_xg = RandomizedSearchCV(xgb, param_distributions=param_grid, cv=5, scoring="f1_weighted", verbose=0, n_jobs=-1, n_iter=100)

In [ ]:
grid_xg.fit(X_train, y_train)

/Users/ori/.pyenv/versions/3.11.3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [12:36:38] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/ori/.pyenv/versions/3.11.3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [12:36:38] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/ori/.pyenv/versions/3.11.3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [12:36:38] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/ori/.pyenv/versions/3.11.3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [12:36:38] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Paramet

,estimator,"XGBClassifier...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.75, 0.8, ...], 'gamma': [0, 0.05], 'learning_rate': [0.04, 0.05, ...], 'max_depth': [5, 6], ...}"
,n_iter,100
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [ ]:
grid_xg.best_params_

{'subsample': 0.85,
 'scale_pos_weight': 1.2,
 'reg_lambda': 1.2,
 'reg_alpha': 0.2,
 'min_child_weight': 1,
 'max_depth': 6,
 'learning_rate': 0.06,
 'gamma': 0,
 'colsample_bytree': 0.8}

In [55]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X_train))

params = {
    "subsample": 0.85,
    "scale_pos_weight": 1.2,
    "reg_lambda": 1.2,
    "reg_alpha": 0.2,
    "min_child_weight": 1,
    "max_depth": 6,
    "learning_rate": 0.06,
    "gamma": 0,
    "colsample_bytree": 0.8,
    "n_estimators": 1200,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "n_jobs": -1
}

for train_idx, val_idx in kf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = XGBClassifier(**params)

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]

In [56]:
meta_X = oof_preds.reshape(-1, 1)
meta_y = y_train

meta_model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1
)

meta_model.fit(meta_X, meta_y)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [61]:
preds = (oof_preds > 0.5).astype(int)

score = f1_score(y_train, preds, average="weighted")

print(score)

0.6957014445242268


In [63]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)
preds = knn.predict(X_test)
print(round(f1_score(y_test, preds),3))


0.483


In [65]:
param_dist_knn = {
    "n_neighbors": [5, 10, 20, 30, 50],
    "weights": ["uniform", "distance"],
    "metric": ["minkowski"],
    "p": [1, 2]
}

In [66]:
grid_knn = RandomizedSearchCV(knn, param_distributions=param_dist_knn, cv=5, scoring="f1_weighted", verbose=0, n_jobs=-1, n_iter=100)

In [ ]:
grid_knn.fit(X_train, y_train)